# 🔢 Notebook 04 – Text Feature Extraction (TF-IDF)
**Goal:** convert cleaned text into numbers that machine-learning models can understand, **without data leakage**.

**Pipeline in this notebook**
1. Understand vectorization and TF-IDF
2. Split data into train / test **first**
3. Fit `TfidfVectorizer` on the **training text only**
4. Explore the vocabulary and the sparse matrix
5. Save the fitted vectorizer

## 1. Understanding Text Vectorization
Machine-learning models work with **numbers**, not words. *Vectorization* converts each text into a list of numbers (a **vector**).
The simplest method is **Bag of Words**: count how many times each word appears. Word order is ignored, but it is fast and works well for sentiment.

## 2. TF-IDF Explained in Simple Language
Counting has a flaw: words like *"flight"* appear in almost every tweet but tell us nothing about sentiment. **TF-IDF** fixes this:

- **TF (Term Frequency):** how often a word appears *in this tweet*. More mentions → more important to this tweet.
- **IDF (Inverse Document Frequency):** how *rare* the word is across *all tweets*. Rare words (like *"rude"*) get a high score; common words get a low score.
- **TF-IDF = TF × IDF** → a word scores high if it is frequent in *this* tweet but rare in *general*.

Finally each tweet vector is **normalised** to length 1, so long and short tweets are comparable.

**Data-leakage warning:** the IDF values are *learned from data*. If we compute them on the whole dataset, information from the test tweets sneaks into training. So we **fit on training text only**, then just *transform* the test text.

## 3. Import Libraries

In [ ]:
# Import TfidfVectorizer and other helpers
import warnings
import joblib
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)

In [ ]:
# Set up project paths (works on your D: drive path, or falls back to the parent of 02_Notebooks)
from pathlib import Path

BASE_DIR = Path(r"D:\DataScience\notebooks\SentimentIQ")
if not BASE_DIR.exists():
    BASE_DIR = Path.cwd().parent          # fallback: notebook is running inside 02_Notebooks/

RAW_PATH       = BASE_DIR / "01_Data" / "Raw" / "Tweets.csv"
PROCESSED_DIR  = BASE_DIR / "01_Data" / "Processed"
CLEANED_PATH   = PROCESSED_DIR / "cleaned_tweets.csv"
MODELS_DIR     = BASE_DIR / "03_Models"
REPORTS_DIR    = BASE_DIR / "05_Reports"

for folder in (PROCESSED_DIR, MODELS_DIR, REPORTS_DIR):
    folder.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42      # same seed everywhere -> reproducible results
TEST_SIZE    = 0.20    # 80% training / 20% testing

print("Project folder :", BASE_DIR)
print("Raw data exists:", RAW_PATH.exists())

## 4. A Tiny TF-IDF Example (to build intuition)

In [ ]:
# Four toy documents: see how TF-IDF scores words
toy_docs = [
    "flight delay terrible",
    "great flight great crew",
    "not happy delay",
    "love great service",
]
toy_vec = TfidfVectorizer()
toy_matrix = toy_vec.fit_transform(toy_docs)

toy_df = pd.DataFrame(toy_matrix.toarray(), columns=toy_vec.get_feature_names_out(),
                      index=[f"doc{i+1}" for i in range(len(toy_docs))]).round(2)
display(toy_df)

idf_df = pd.DataFrame({"word": toy_vec.get_feature_names_out(), "idf": toy_vec.idf_.round(2)}).sort_values("idf")
display(idf_df.T)

**Read it like this:** `great` appears twice in doc2 (high TF). `flight` and `delay` appear in two documents (lower IDF), while `terrible` appears in only one (higher IDF).

## 5. Load the Cleaned Dataset

In [ ]:
# Load the file created in Notebook 03
try:
    df = pd.read_csv(CLEANED_PATH)
except FileNotFoundError:
    raise FileNotFoundError("cleaned_tweets.csv not found. Please run Notebook 03 first.")

df = df.dropna(subset=["cleaned_text", "airline_sentiment"]).reset_index(drop=True)
print("Shape:", df.shape)
df[["cleaned_text", "airline_sentiment"]].head()

## 6. Split BEFORE Vectorizing (prevents leakage)

In [ ]:
# Define X (input) and y (target) and split 80/20, keeping the class proportions identical in both parts
X = df["cleaned_text"]
y = df["airline_sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

print(f"Training tweets: {len(X_train):,}")
print(f"Testing tweets : {len(X_test):,}")
print("\nTrain class share (%):")
print((y_train.value_counts(normalize=True) * 100).round(1))
print("\nTest class share (%):")
print((y_test.value_counts(normalize=True) * 100).round(1))

In [ ]:
# Save the split so it can be inspected later (Notebook 05 recreates it with the same seed)
train_out = pd.DataFrame({"cleaned_text": X_train.values, "airline_sentiment": y_train.values})
test_out  = pd.DataFrame({"cleaned_text": X_test.values,  "airline_sentiment": y_test.values})
train_out.to_csv(PROCESSED_DIR / "train.csv", index=False)
test_out.to_csv(PROCESSED_DIR / "test.csv", index=False)
print("✅ Saved train.csv and test.csv in", PROCESSED_DIR)

## 7. Convert Cleaned Text into TF-IDF Features

In [ ]:
# Create the vectorizer and learn the vocabulary + IDF from the TRAINING text only
tfidf = TfidfVectorizer(
    max_features=10000,     # keep the 10,000 most useful terms
    ngram_range=(1, 2),     # single words AND two-word phrases (captures "not good", "customer service")
    min_df=2,               # ignore terms that appear in fewer than 2 tweets (typos, noise)
    max_df=0.95,            # ignore terms that appear in more than 95% of tweets
    sublinear_tf=True,      # use 1 + log(tf) so very repeated words do not dominate
)

X_train_tfidf = tfidf.fit_transform(X_train)   # fit + transform on training data
X_test_tfidf  = tfidf.transform(X_test)        # ONLY transform on test data (no fitting!)

print("Training matrix shape:", X_train_tfidf.shape)
print("Testing matrix shape :", X_test_tfidf.shape)

**Expected output:** the number of columns is the same for train and test (≤ 10,000) – each column is one word or phrase.

## 8. Explore the Vocabulary

In [ ]:
# Vocabulary size and the split between single words and two-word phrases
features = tfidf.get_feature_names_out()
unigrams = [f for f in features if " " not in f]
bigrams  = [f for f in features if " " in f]

print(f"Total vocabulary size : {len(features):,}")
print(f"Single words (unigrams): {len(unigrams):,}")
print(f"Two-word phrases       : {len(bigrams):,}")
print("\nRandom sample of features:")
print(list(np.random.RandomState(RANDOM_STATE).choice(features, 25, replace=False)))

In [ ]:
# Which terms are the most common (lowest IDF) and which are the rarest (highest IDF)?
idf_table = pd.DataFrame({"term": features, "idf": tfidf.idf_}).sort_values("idf")
print("Most common terms (low IDF):")
display(idf_table.head(10).reset_index(drop=True))
print("Rarest terms (high IDF):")
display(idf_table.tail(10).reset_index(drop=True))

In [ ]:
# Do negation phrases survive in the vocabulary?
negation_phrases = [f for f in bigrams if f.startswith(("not ", "no ", "never "))]
print(f"{len(negation_phrases)} negation phrases found, e.g.:")
print(negation_phrases[:20])

## 9. Display Sample TF-IDF Features

In [ ]:
# For three training tweets, list their non-zero TF-IDF terms with weights
for i in range(3):
    row = X_train_tfidf[i]
    terms = [(features[j], round(row[0, j], 3)) for j in row.indices]
    terms.sort(key=lambda t: t[1], reverse=True)
    print(f"Tweet {i+1}: {X_train.iloc[i]}")
    print(f"   label = {y_train.iloc[i]}")
    print(f"   top TF-IDF terms: {terms[:6]}\n")

In [ ]:
# A small slice of the matrix as a table (5 tweets x 10 columns); most cells are 0
slice_df = pd.DataFrame(X_train_tfidf[:5, :10].toarray(), columns=features[:10]).round(3)
slice_df

## 10. Understanding the Sparse Matrix

In [ ]:
# TF-IDF output is a SPARSE matrix: it stores only the non-zero values, which saves lots of memory
n_rows, n_cols = X_train_tfidf.shape
non_zero = X_train_tfidf.nnz
density = non_zero / (n_rows * n_cols) * 100
dense_mb  = n_rows * n_cols * 8 / 1024**2
sparse_mb = (X_train_tfidf.data.nbytes + X_train_tfidf.indices.nbytes + X_train_tfidf.indptr.nbytes) / 1024**2

print("Matrix type          :", type(X_train_tfidf).__name__)
print(f"Shape                : {n_rows:,} tweets x {n_cols:,} features")
print(f"Non-zero values      : {non_zero:,}")
print(f"Density              : {density:.3f}%  (so {100 - density:.2f}% of cells are zero)")
print(f"Memory if dense      : {dense_mb:,.1f} MB")
print(f"Memory as sparse     : {sparse_mb:,.1f} MB")
print(f"Avg terms per tweet  : {non_zero / n_rows:.1f}")

In [ ]:
# Each row is normalised to length 1 (L2 norm), so the vector length of every tweet should be ~1.0
row_norms = np.sqrt(X_train_tfidf.multiply(X_train_tfidf).sum(axis=1)).A1
print("Row norms -> min: %.3f | max: %.3f" % (row_norms.min(), row_norms.max()))
print("(rows that are all-zero have norm 0: tweets whose words are not in the vocabulary)")
print("All-zero rows in train:", int((row_norms == 0).sum()))

**In simple words:** the sparse matrix is a giant table – one row per tweet, one column per word/phrase – where most cells are 0. Python stores only the non-zero cells, which keeps memory small and training fast.

## 11. Save the Fitted Vectorizer

In [ ]:
# Save the vectorizer (it contains the vocabulary and IDF weights learned from training data)
vectorizer_path = MODELS_DIR / "tfidf_vectorizer.pkl"
try:
    joblib.dump(tfidf, vectorizer_path)
    print("✅ Saved:", vectorizer_path, f"({vectorizer_path.stat().st_size / 1024:.0f} KB)")
except Exception as e:
    print("❌ Could not save the vectorizer:", e)

# Quick reload test
reloaded = joblib.load(vectorizer_path)
assert reloaded.transform(["not good delay"]).shape[1] == len(features)
print("Reload test passed.")

> ℹ️ Notebook 07 will tune the vectorizer settings and Notebook 08 will save the **final** matching pair `sentiment_model.pkl` + `tfidf_vectorizer.pkl`. A model must always be used with the vectorizer it was trained with.

**Next:** Notebook 05 – Model Training.